# OP-03 · Cumuls C3S par période (décades, mois, saisons)

**Notebook opérationnel** — à exécuter chaque mois, après `OP_01`.

| | |
|---|---|
| Étape du workflow | E2 → entrée de E4 et E5 |
| Entrée | fichiers bruts `DATA_OSF/raw/c3s/YYYYMM/` |
| Sorties | `DATA_OSF/derived/c3s/YYYYMM/c3s_<centre>_precip_{forecast,hindcast}_periods.nc` + tableau récapitulatif |
| Durée | environ 4 min pour 7 modèles |

La pluie journalière est attribuée au jour où elle tombe (jour 0 = jour d'initialisation). Seules les périodes **complètes** dans l'horizon reçu de chaque modèle sont calculées.

Équivalent en ligne de commande : `python scripts/run_c3s_totals.py --config config/cycle_YYYYMM.yaml`

## Paramètres

In [ ]:
CYCLE_CONFIG = "config/cycle_202609.yaml"
MODELS       = None      # None = tous les modèles de la configuration

In [ ]:
from pathlib import Path
import os
import pandas as pd

REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(REPO)
from eccas_s2s.settings import load_cycle
cfg = load_cycle(CYCLE_CONFIG)
print(f"Cycle {cfg.cycle_id} — initialisation {cfg.init_date.date()}")

## 1. Calcul des cumuls

In [ ]:
from eccas_s2s.operations import c3s_totals
ctx = c3s_totals.run(CYCLE_CONFIG, models=MODELS)
print(f"\nStatut : {ctx.status}")
for w in ctx.warnings:
    print(" ⚠", w)

## 2. Récapitulatif

In [ ]:
summary = pd.DataFrame(ctx.parameters["summary"])
summary

## 3. Contrôle visuel : cumul de la première saison, moyenne d'ensemble

In [ ]:
from eccas_s2s.viz.maps import map_panel
fields, titles = [], []
for c in summary.loc[summary.kind == "forecast", "centre"]:
    da = c3s_totals.load_totals(cfg, c, "forecast").sel(period="season_m0").isel(year=0)
    fields.append(da.mean("number")); titles.append(f"{cfg.c3s_models[c].label} ({da.sizes['number']} m.)")
fig = map_panel(fields, titles, shapefile=cfg.raw["paths"]["shapefile"], cmap="YlGnBu",
                levels=[0, 50, 100, 200, 300, 400, 500, 700, 900, 1200], extend="max", ncols=4,
                cbar_label="mm", suptitle=f"{str(da['label'].values)} — cumul brut, moyenne d'ensemble")

In [ ]:
print("manifeste :", ctx.run_dir / "manifest.json")